In [6]:
pip install reportlab matplotlib --break-system-packages -q

In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import io

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm, mm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY, TA_LEFT, TA_RIGHT
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    HRFlowable, PageBreak, Image, KeepTogether
)
from reportlab.platypus.flowables import HRFlowable
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

W, H = A4

# ── colour palette ──────────────────────────────────────────────────────────
DARK_GREEN  = colors.HexColor('#1B4332')
MID_GREEN   = colors.HexColor('#2D6A4F')
LIGHT_GREEN = colors.HexColor('#95D5B2')
PALE_GREEN  = colors.HexColor('#D8F3DC')
BLUE        = colors.HexColor('#185FA5')
AMBER       = colors.HexColor('#854F0B')
BROWN       = colors.HexColor('#3B2F2F')
LIGHT_GRAY  = colors.HexColor('#F4F4F4')
MID_GRAY    = colors.HexColor('#CCCCCC')
DARK_GRAY   = colors.HexColor('#444444')
WHITE       = colors.white
BLACK       = colors.black

# ── styles ───────────────────────────────────────────────────────────────────
styles = getSampleStyleSheet()

def make_style(name, parent='Normal', **kwargs):
    return ParagraphStyle(name, parent=styles[parent], **kwargs)

cover_title   = make_style('CoverTitle',   fontSize=26, leading=32, textColor=WHITE,
                            alignment=TA_CENTER, spaceAfter=10, fontName='Helvetica-Bold')
cover_sub     = make_style('CoverSub',     fontSize=13, leading=18, textColor=LIGHT_GREEN,
                            alignment=TA_CENTER, spaceAfter=6,  fontName='Helvetica')
cover_meta    = make_style('CoverMeta',    fontSize=10, leading=14, textColor=WHITE,
                            alignment=TA_CENTER, fontName='Helvetica')

h1 = make_style('H1', fontSize=15, leading=20, textColor=DARK_GREEN, spaceBefore=18,
                 spaceAfter=6, fontName='Helvetica-Bold', borderPad=0)
h2 = make_style('H2', fontSize=12, leading=16, textColor=MID_GREEN,  spaceBefore=12,
                 spaceAfter=4, fontName='Helvetica-Bold')
h3 = make_style('H3', fontSize=10.5, leading=14, textColor=BROWN,    spaceBefore=8,
                 spaceAfter=3, fontName='Helvetica-Bold')

body = make_style('Body', fontSize=10, leading=15, textColor=DARK_GRAY,
                   alignment=TA_JUSTIFY, spaceAfter=6)
body_small = make_style('BodySm', fontSize=9, leading=13, textColor=DARK_GRAY,
                          alignment=TA_JUSTIFY, spaceAfter=4)
caption = make_style('Caption', fontSize=8.5, leading=12, textColor=colors.HexColor('#666666'),
                      alignment=TA_CENTER, spaceAfter=8, fontName='Helvetica-Oblique')
ref_style = make_style('Ref', fontSize=8.5, leading=13, textColor=DARK_GRAY,
                         spaceAfter=3, leftIndent=14, firstLineIndent=-14)
abstract_style = make_style('Abstract', fontSize=9.5, leading=14.5,
                              textColor=DARK_GRAY, alignment=TA_JUSTIFY,
                              leftIndent=20, rightIndent=20, spaceAfter=4)
kw_style = make_style('KW', fontSize=9, leading=13, textColor=MID_GREEN,
                        leftIndent=20, spaceAfter=10, fontName='Helvetica-Oblique')
toc_style = make_style('TOC', fontSize=10, leading=18, textColor=DARK_GRAY)
toc_h = make_style('TOCH', fontSize=11, leading=20, textColor=DARK_GREEN,
                    fontName='Helvetica-Bold', spaceAfter=4)

# ── chart generator ──────────────────────────────────────────────────────────
def make_chart():
    fig, ax = plt.subplots(figsize=(7.5, 4.2))
    fig.patch.set_facecolor('#FAFAFA')
    ax.set_facecolor('#FAFAFA')

    strategies = [
        'Feed Additives\n(Cattle)',
        'Manure Mgmt.\n(Biogas)',
        'Alt. Wetting\n& Drying',
        'Fertilizer\nOptimization',
        'Agroforestry &\nC Sequestration'
    ]
    mid   = [25, 65, 50, 35, 33]
    lo    = [20, 50, 30, 20, 15]
    hi    = [30, 80, 70, 50, 50]
    sec   = [ 0,  0,  0, 10, 15]
    gases = ['CH\u2084', 'CH\u2084', 'CH\u2084', 'N\u2082O', 'CO\u2082']
    prim_colors  = ['#185FA5','#185FA5','#185FA5','#854F0B','#2D6A4F']
    sec_colors   = ['none',   'none',   'none',   '#7EB8E8','#95D5B2']

    x = np.arange(len(strategies))
    w = 0.38

    bars1 = ax.bar(x, mid, width=w, color=prim_colors, zorder=3,
                   label='Primary gas', linewidth=0, edgecolor='none')

    for i, (xi, lo_i, hi_i) in enumerate(zip(x, lo, hi)):
        ax.plot([xi, xi], [lo_i, hi_i], color='#222', lw=1.4, zorder=4)
        ax.plot([xi-0.07, xi+0.07], [lo_i, lo_i], color='#222', lw=1.4, zorder=4)
        ax.plot([xi-0.07, xi+0.07], [hi_i, hi_i], color='#222', lw=1.4, zorder=4)

    for i, (xi, sv, sc) in enumerate(zip(x, sec, sec_colors)):
        if sv > 0:
            ax.bar(xi+w, sv, width=w*0.8, color=sc, zorder=3,
                   linewidth=0.6, edgecolor='#aaa')

    for i, (xi, mv, gas) in enumerate(zip(x, mid, gases)):
        ax.text(xi, mv+1.5, gas, ha='center', va='bottom',
                fontsize=8, color=prim_colors[i], fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(strategies, fontsize=8.2)
    ax.set_ylabel('Estimated GHG Emission Reduction (%)', fontsize=9)
    ax.set_ylim(0, 92)
    ax.set_xlim(-0.55, len(strategies)-0.35)
    ax.yaxis.grid(True, linestyle='--', alpha=0.5, linewidth=0.5)
    ax.set_axisbelow(True)
    for sp in ['top','right']:
        ax.spines[sp].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    p1 = mpatches.Patch(color='#185FA5', label='CH\u2084 (primary)')
    p2 = mpatches.Patch(color='#854F0B', label='N\u2082O (primary)')
    p3 = mpatches.Patch(color='#2D6A4F', label='CO\u2082 (primary)')
    p4 = mpatches.Patch(color='#7EB8E8', label='CH\u2084 (secondary)')
    p5 = mpatches.Patch(color='#95D5B2', label='N\u2082O (secondary)')
    ax.legend(handles=[p1,p2,p3,p4,p5], fontsize=7.5, loc='upper right',
              framealpha=0.7, edgecolor='#ccc', ncol=2)

    plt.tight_layout(pad=0.5)
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=180, bbox_inches='tight')
    plt.close()
    buf.seek(0)
    return buf

def make_radar():
    fig, ax = plt.subplots(figsize=(4.5, 4.5), subplot_kw=dict(polar=True))
    fig.patch.set_facecolor('#FAFAFA')

    cats = ['Emission\nReduction', 'Cost\nEfficiency', 'Scalability',
            'Co-benefits', 'Adoption\nEase', 'Data\nCertainty']
    N = len(cats)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]

    datasets = {
        'Feed Additives':        [7, 5, 7, 4, 6, 8],
        'Manure Biogas':         [9, 6, 6, 8, 5, 8],
        'Alt. Wetting & Drying': [7, 8, 8, 5, 7, 7],
        'Fertilizer Optim.':     [6, 7, 9, 6, 8, 7],
        'Agroforestry':          [6, 5, 6, 9, 6, 6],
    }
    palette = ['#185FA5','#2D6A4F','#854F0B','#C47F17','#9B2335']

    for (label, vals), col in zip(datasets.items(), palette):
        vals_plot = vals + vals[:1]
        ax.plot(angles, vals_plot, 'o-', linewidth=1.4, color=col, label=label, markersize=3)
        ax.fill(angles, vals_plot, alpha=0.07, color=col)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(cats, fontsize=7.5)
    ax.set_ylim(0, 10)
    ax.set_yticks([2,4,6,8,10])
    ax.set_yticklabels(['2','4','6','8','10'], fontsize=6.5, color='gray')
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15),
              fontsize=7, framealpha=0.7)
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=180, bbox_inches='tight')
    plt.close()
    buf.seek(0)
    return buf

# ── page template with header/footer ─────────────────────────────────────────
page_num = [0]

def on_page(canvas, doc):
    page_num[0] += 1
    canvas.saveState()
    if page_num[0] > 1:
        # header bar
        canvas.setFillColor(DARK_GREEN)
        canvas.rect(1.5*cm, H-1.2*cm, W-3*cm, 0.4*cm, fill=1, stroke=0)
        canvas.setFillColor(WHITE)
        canvas.setFont('Helvetica', 7)
        canvas.drawString(1.6*cm, H-1.0*cm, 'GHG Mitigation in Livestock & Agriculture')
        canvas.drawRightString(W-1.6*cm, H-1.0*cm, 'Scientific Assignment | 2025')
        # footer
        canvas.setFillColor(MID_GRAY)
        canvas.rect(1.5*cm, 0.8*cm, W-3*cm, 0.02*cm, fill=1, stroke=0)
        canvas.setFillColor(colors.HexColor('#888'))
        canvas.setFont('Helvetica', 7.5)
        canvas.drawCentredString(W/2, 0.55*cm, f'Page {page_num[0]}')
    canvas.restoreState()

def on_first_page(canvas, doc):
    pass  # cover page handles its own drawing

# ── cover page (drawn directly on canvas) ────────────────────────────────────
class CoverPage:
    def __init__(self):
        self.width = W
        self.height = H

    def draw(self, canvas, doc):
        c = canvas
        # dark green background top band
        c.setFillColor(DARK_GREEN)
        c.rect(0, H*0.42, W, H*0.58, fill=1, stroke=0)

        # decorative stripe
        c.setFillColor(MID_GREEN)
        c.rect(0, H*0.42, W, 0.8*cm, fill=1, stroke=0)

        # pale bottom band
        c.setFillColor(PALE_GREEN)
        c.rect(0, 0, W, H*0.42, fill=1, stroke=0)

        # left accent bar
        c.setFillColor(LIGHT_GREEN)
        c.rect(1.5*cm, H*0.28, 0.35*cm, H*0.20, fill=1, stroke=0)

        # title
        c.setFillColor(WHITE)
        c.setFont('Helvetica-Bold', 22)
        c.drawCentredString(W/2, H*0.74, 'Greenhouse Gas Emission Mitigation')
        c.setFont('Helvetica-Bold', 22)
        c.drawCentredString(W/2, H*0.69, 'Strategies in Livestock and Agriculture')

        c.setFillColor(LIGHT_GREEN)
        c.setFont('Helvetica', 12)
        c.drawCentredString(W/2, H*0.63, 'A Comparative Analysis of Major Interventions')
        c.drawCentredString(W/2, H*0.60, 'and Their Efficacy in Reducing CH\u2084, N\u2082O, and CO\u2082 Emissions')

        # separator line
        c.setStrokeColor(LIGHT_GREEN)
        c.setLineWidth(0.5)
        c.line(3*cm, H*0.57, W-3*cm, H*0.57)

        # metadata block
        c.setFillColor(WHITE)
        c.setFont('Helvetica', 10)
        meta_lines = [
            ('Course:', 'Environmental Science & Sustainable Agriculture'),
            ('Subject:', 'Climate Change Mitigation in Food Systems'),
            ('Submitted by:', 'Your Name  |  Student ID: XXXXXXX'),
            ('Instructor:', 'Prof. [Name]  |  Dept. of Environmental Science'),
            ('Date:', 'May 2025'),
        ]
        y = H*0.53
        for label, val in meta_lines:
            c.setFont('Helvetica-Bold', 9)
            c.drawString(3.5*cm, y, label)
            c.setFont('Helvetica', 9)
            c.drawString(6.5*cm, y, val)
            y -= 0.55*cm

        # bottom section – abstract teaser
        c.setFillColor(DARK_GRAY)
        c.setFont('Helvetica-Bold', 10)
        c.drawString(2.2*cm, H*0.36, 'ABSTRACT OVERVIEW')
        c.setFillColor(MID_GREEN)
        c.rect(2.2*cm, H*0.355, 4*cm, 0.04*cm, fill=1, stroke=0)

        abstract_text = (
            "Agriculture accounts for 10–12% of global anthropogenic greenhouse gas emissions. "
            "This assignment critically examines five evidence-based mitigation strategies — feed additives, "
            "manure management, alternate wetting and drying, fertilizer optimization, and agroforestry — "
            "evaluating their emission-reduction potential, co-benefits, scalability, and adoption barriers."
        )
        from reportlab.platypus import Paragraph
        from reportlab.lib.styles import ParagraphStyle
        from reportlab.lib.enums import TA_JUSTIFY
        abs_s = ParagraphStyle('abs_cover', fontSize=9, leading=14, textColor=DARK_GRAY,
                                alignment=TA_JUSTIFY)
        p = Paragraph(abstract_text, abs_s)
        p.wrapOn(c, W-4.4*cm, 5*cm)
        p.drawOn(c, 2.2*cm, H*0.16)

        # page number omitted on cover

# ── document builder ──────────────────────────────────────────────────────────
def build():
    out = '/mnt/user-data/outputs/GHG_Mitigation_Agriculture_Assignment.pdf'
    doc = SimpleDocTemplate(
        out, pagesize=A4,
        leftMargin=2.2*cm, rightMargin=2.2*cm,
        topMargin=2.2*cm,  bottomMargin=2.2*cm,
        title='GHG Emission Mitigation in Livestock and Agriculture',
        author='Scientific Assignment'
    )

    story = []

    # ── COVER (full-page canvas item) ─────────────────────────────────────────
    from reportlab.platypus import Flowable
    class CoverFlowable(Flowable):
        def __init__(self):
            Flowable.__init__(self)
            self.width = W
            self.height = H
        def draw(self):
            cp = CoverPage()
            cp.draw(self.canv, None)

    story.append(CoverFlowable())
    story.append(PageBreak())

    # ── TABLE OF CONTENTS ──────────────────────────────────────────────────────
    story.append(Paragraph('Table of Contents', h1))
    story.append(HRFlowable(width='100%', thickness=1, color=LIGHT_GREEN, spaceAfter=8))

    toc_entries = [
        ('1.', 'Introduction', '3'),
        ('2.', 'Background: Agriculture and Climate Change', '3'),
        ('3.', 'Mitigation Strategy 1: Feed Additives for Enteric Fermentation', '4'),
        ('4.', 'Mitigation Strategy 2: Manure Management Systems', '5'),
        ('5.', 'Mitigation Strategy 3: Alternate Wetting and Drying in Rice Cultivation', '6'),
        ('6.', 'Mitigation Strategy 4: Fertilizer Optimization', '7'),
        ('7.', 'Mitigation Strategy 5: Agroforestry and Carbon Sequestration', '8'),
        ('8.', 'Comparative Analysis', '9'),
        ('9.', 'Challenges and Barriers to Implementation', '10'),
        ('10.', 'Policy Recommendations', '10'),
        ('11.', 'Conclusion', '11'),
        ('12.', 'References', '11'),
    ]
    toc_data = [[Paragraph(f'<b>{n}</b>', toc_style),
                 Paragraph(t, toc_style),
                 Paragraph(f'<font color="#2D6A4F">{pg}</font>', toc_style)]
                for n, t, pg in toc_entries]
    toc_table = Table(toc_data, colWidths=[1*cm, 13.2*cm, 1*cm])
    toc_table.setStyle(TableStyle([
        ('VALIGN', (0,0), (-1,-1), 'TOP'),
        ('BOTTOMPADDING', (0,0), (-1,-1), 5),
        ('TOPPADDING', (0,0), (-1,-1), 2),
        ('LINEBELOW', (0,0), (-1,-2), 0.3, MID_GRAY),
        ('ROWBACKGROUNDS', (0,0), (-1,-1), [WHITE, LIGHT_GRAY]),
    ]))
    story.append(toc_table)
    story.append(PageBreak())

    # ── 1. INTRODUCTION ────────────────────────────────────────────────────────
    story.append(Paragraph('1. Introduction', h1))
    story.append(HRFlowable(width='100%', thickness=0.5, color=LIGHT_GREEN, spaceAfter=6))
    story.append(Paragraph(
        'Climate change represents one of the most pressing challenges of the 21st century. '
        'The agricultural sector, encompassing crop production, livestock rearing, and land-use '
        'change, contributes approximately 10–12% of total global anthropogenic greenhouse gas (GHG) '
        'emissions (IPCC, 2022). These emissions span all three primary GHGs: methane (CH<sub>4</sub>), '
        'nitrous oxide (N<sub>2</sub>O), and carbon dioxide (CO<sub>2</sub>), each with distinct '
        'sources, global warming potentials (GWPs), and mitigation pathways.', body))
    story.append(Paragraph(
        'The urgency of reducing agricultural GHG emissions is compounded by simultaneous demands '
        'for increased food production to sustain a projected global population of 9.7 billion by '
        '2050 (FAO, 2020). This dual challenge — decarbonising food systems while scaling production '
        '— necessitates the identification and deployment of high-efficacy, cost-effective mitigation '
        'strategies.', body))
    story.append(Paragraph(
        'This assignment critically evaluates five major mitigation strategies: (1) feed additives to '
        'reduce enteric methane, (2) improved manure management systems, (3) alternate wetting and '
        'drying (AWD) in paddy rice cultivation, (4) precision fertilizer management and nitrification '
        'inhibitors, and (5) agroforestry and carbon sequestration. For each strategy, the assignment '
        'reviews the underlying mechanisms, empirical evidence for emission reductions, co-benefits, '
        'implementation challenges, and scalability.', body))

    # ── 2. BACKGROUND ──────────────────────────────────────────────────────────
    story.append(Paragraph('2. Background: Agriculture and Climate Change', h1))
    story.append(HRFlowable(width='100%', thickness=0.5, color=LIGHT_GREEN, spaceAfter=6))
    story.append(Paragraph(
        'The agricultural sector emits GHGs through multiple pathways. Livestock enteric fermentation '
        'is the single largest agricultural source of CH<sub>4</sub>, representing ~14.5% of global '
        'GHG emissions from the livestock sector alone (Gerber et al., 2013). Manure decomposition '
        'under anaerobic conditions generates both CH<sub>4</sub> and N<sub>2</sub>O. Flooded paddy '
        'rice fields create ideal anaerobic environments for methanogenic archaea, producing substantial '
        'CH<sub>4</sub>. Nitrogen fertilizers, both synthetic and organic, promote microbial '
        'nitrification and denitrification processes that release N<sub>2</sub>O — a gas with a '
        '100-year GWP of 273 relative to CO<sub>2</sub> (IPCC AR6, 2022).', body))

    # GHG emissions table
    story.append(Paragraph('Table 1: Key GHG Sources in Agriculture and Their Global Warming Potential', h3))
    table1_data = [
        ['GHG', 'Primary Agricultural Source', 'GWP (100-yr)', '% of Agri. Emissions'],
        ['CH\u2084', 'Enteric fermentation, paddy rice, manure', '27.9', '~50%'],
        ['N\u2082O', 'Synthetic fertilizers, manure, soil', '273', '~35%'],
        ['CO\u2082', 'Land-use change, fossil fuel use', '1', '~15%'],
    ]
    t1 = Table(table1_data, colWidths=[2.5*cm, 7.5*cm, 2.8*cm, 3.5*cm])
    t1.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), DARK_GREEN),
        ('TEXTCOLOR',  (0,0), (-1,0), WHITE),
        ('FONTNAME',   (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTSIZE',   (0,0), (-1,-1), 9),
        ('ROWBACKGROUNDS', (0,1), (-1,-1), [WHITE, PALE_GREEN]),
        ('GRID', (0,0), (-1,-1), 0.4, MID_GRAY),
        ('ALIGN', (2,0), (-1,-1), 'CENTER'),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('TOPPADDING', (0,0), (-1,-1), 5),
        ('BOTTOMPADDING', (0,0), (-1,-1), 5),
    ]))
    story.append(t1)
    story.append(Paragraph('Source: IPCC AR6 (2022); Gerber et al. (2013)', caption))
    story.append(Spacer(1, 0.3*cm))

    story.append(Paragraph(
        'Land-use change, particularly deforestation for pasture or cropland expansion, constitutes '
        'an additional significant source of CO<sub>2</sub> release. Together, these pathways create '
        'a complex emissions profile that requires multi-pronged mitigation approaches tailored to '
        'specific agricultural systems, geographies, and socioeconomic contexts.', body))
    story.append(PageBreak())

    # ── 3. FEED ADDITIVES ──────────────────────────────────────────────────────
    story.append(Paragraph('3. Mitigation Strategy 1: Feed Additives for Enteric Fermentation Reduction', h1))
    story.append(HRFlowable(width='100%', thickness=0.5, color=LIGHT_GREEN, spaceAfter=6))
    story.append(Paragraph('3.1 Mechanism', h2))
    story.append(Paragraph(
        'Enteric fermentation occurs in the rumen of cattle, sheep, and other ruminants, where '
        'microbial communities digest fibrous feed materials. Methanogenic archaea — primarily '
        '<i>Methanobrevibacter ruminantium</i> — utilize hydrogen produced during fermentation to '
        'reduce CO<sub>2</sub> to CH<sub>4</sub>, which is expelled via eructation. Several feed '
        'additive classes have been developed to inhibit this process.', body))
    story.append(Paragraph(
        '3-nitrooxypropanol (3-NOP), marketed as Bovaer®, directly inhibits methyl-coenzyme M '
        'reductase (MCR), the enzyme catalysing the final step of methanogenesis. Clinical trials '
        'have demonstrated consistent reductions of 20–30% in enteric CH<sub>4</sub> across beef '
        'and dairy systems without adversely affecting animal productivity (Hristov et al., 2015). '
        'Bromoform-containing seaweeds (<i>Asparagopsis taxiformis</i>) have demonstrated even '
        'larger reductions — up to 80% in controlled settings — though field-scale results vary '
        'between 20–50% depending on dose and feed composition.', body))
    story.append(Paragraph('3.2 Efficacy and Evidence', h2))
    story.append(Paragraph(
        'A meta-analysis by Beauchemin et al. (2020) across 52 trials found that lipid-based '
        'additives reduce enteric CH<sub>4</sub> by 15–20%, while 3-NOP achieves 20–30% across '
        'diverse production systems. Ionophores such as monensin offer modest reductions (5–10%) '
        'but are already widely used in beef feedlots. The interaction between additive type, '
        'dose, feed composition, and animal species introduces variability in observed outcomes.', body))
    story.append(Paragraph('3.3 Co-benefits and Limitations', h2))
    story.append(Paragraph(
        'Methane suppression redirects hydrogen toward propionate production, slightly improving '
        'feed energy use efficiency. However, persistent efficacy in pasture-based systems remains '
        'challenging due to delivery logistics. Cost (USD 0.10–0.30 per animal per day for 3-NOP) '
        'and regulatory approval timelines limit rapid global scaling.', body))

    # info box
    info_data = [[Paragraph(
        '<b>Key Finding:</b> Feed additives targeting enteric fermentation offer a rapid, '
        'farm-scalable CH<sub>4</sub> reduction of 20–30%, making them one of the most '
        'technologically mature mitigation options for the livestock sector.',
        body_small)]]
    info_table = Table(info_data, colWidths=[W-4.4*cm])
    info_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), PALE_GREEN),
        ('LEFTPADDING', (0,0), (-1,-1), 10),
        ('RIGHTPADDING', (0,0), (-1,-1), 10),
        ('TOPPADDING', (0,0), (-1,-1), 8),
        ('BOTTOMPADDING', (0,0), (-1,-1), 8),
        ('BOX', (0,0), (-1,-1), 1, MID_GREEN),
        ('LINEAFTER', (0,0), (0,-1), 4, DARK_GREEN),
    ]))
    story.append(info_table)
    story.append(Spacer(1, 0.4*cm))

    # ── 4. MANURE MANAGEMENT ──────────────────────────────────────────────────
    story.append(Paragraph('4. Mitigation Strategy 2: Manure Management Systems', h1))
    story.append(HRFlowable(width='100%', thickness=0.5, color=LIGHT_GREEN, spaceAfter=6))
    story.append(Paragraph('4.1 Mechanism', h2))
    story.append(Paragraph(
        'Livestock manure stored in anaerobic lagoons or piles undergoes microbial decomposition, '
        'generating substantial CH<sub>4</sub> and N<sub>2</sub>O. Anaerobic digestion (AD) systems '
        'capture the biogas generated during controlled decomposition, preventing its atmospheric '
        'release while producing renewable energy. Composting under aerobic conditions dramatically '
        'reduces CH<sub>4</sub> production while partially oxidising nitrogen compounds.', body))
    story.append(Paragraph('4.2 Efficacy and Evidence', h2))
    story.append(Paragraph(
        'Covered anaerobic lagoons with biogas capture achieve 50–80% reduction in CH<sub>4</sub> '
        'emissions from manure storage (Amon et al., 2006). The captured biogas — typically '
        '60–70% CH<sub>4</sub> — can be combusted for heat and electricity or upgraded to '
        'biomethane for grid injection. Life-cycle analyses demonstrate that biogas systems can '
        'achieve net-negative GHG balances when displacing fossil fuel electricity (Weiland, 2010). '
        'Composting reduces CH<sub>4</sub> by 40–60% relative to anaerobic storage, though it may '
        'modestly increase N<sub>2</sub>O if moisture and aeration are not well managed.', body))
    story.append(Paragraph('4.3 Co-benefits and Economic Considerations', h2))
    story.append(Paragraph(
        'Biogas systems offer multiple co-benefits: renewable energy generation, digestate '
        'as a higher-quality fertiliser, reduced odour, and pathogen reduction. Economic '
        'feasibility depends strongly on farm scale, energy prices, and policy incentives. '
        'Payback periods of 5–10 years are typical for medium-to-large livestock operations '
        'in OECD countries, while smaller farms in developing nations may require cooperative '
        'models or subsidised financing (FAO, 2020).', body))
    story.append(PageBreak())

    # ── 5. AWD ────────────────────────────────────────────────────────────────
    story.append(Paragraph('5. Mitigation Strategy 3: Alternate Wetting and Drying in Rice Cultivation', h1))
    story.append(HRFlowable(width='100%', thickness=0.5, color=LIGHT_GREEN, spaceAfter=6))
    story.append(Paragraph('5.1 Mechanism', h2))
    story.append(Paragraph(
        'Continuously flooded paddy rice fields create persistent anaerobic soil conditions '
        'that support methanogenic microbial communities. Global rice cultivation contributes '
        'approximately 9–11% of total agricultural CH<sub>4</sub> emissions (Saunois et al., 2020). '
        'Alternate Wetting and Drying (AWD) is an irrigation management technique where fields '
        'are allowed to dry for several days between irrigation events, periodically introducing '
        'aerobic soil conditions that suppress methanogenesis.', body))
    story.append(Paragraph('5.2 Efficacy and Evidence', h2))
    story.append(Paragraph(
        'Field trials across Asia — the dominant rice-producing region — demonstrate CH<sub>4</sub> '
        'reductions of 30–70% with AWD, with variation attributable to soil type, climate, rice '
        'variety, and the degree of drying applied (Linquist et al., 2015). A comprehensive '
        'meta-analysis by Lagomarsino et al. (2016) across 57 datasets found a mean reduction '
        'of 48% in CH<sub>4</sub> emissions under AWD compared to continuous flooding, with '
        'minimal yield penalty when safe AWD (maintaining soil water potential above -20 kPa) '
        'is applied.', body))
    story.append(Paragraph(
        'An important trade-off exists: drying soils accelerates nitrification and denitrification, '
        'potentially increasing N<sub>2</sub>O emissions by 10–20% (Kritee et al., 2018). '
        'However, since N<sub>2</sub>O has a much higher GWP (273) than CH<sub>4</sub> (27.9), '
        'on a CO<sub>2</sub>-equivalent basis the net effect remains strongly positive when '
        'AWD is well-managed.', body))
    story.append(Paragraph('5.3 Adoption and Water Savings', h2))
    story.append(Paragraph(
        'AWD offers the dual advantage of reducing water use by 25–40% alongside emission '
        'reductions — critically important in water-scarce regions. Farmer adoption has been '
        'facilitated through low-tech field water tubes for monitoring soil water levels. '
        'Large-scale programmes in Bangladesh, Vietnam, and the Philippines have demonstrated '
        'feasibility at national scale with appropriate extension services.', body))

    # ── 6. FERTILIZER OPTIMIZATION ────────────────────────────────────────────
    story.append(Paragraph('6. Mitigation Strategy 4: Fertilizer Optimization', h1))
    story.append(HRFlowable(width='100%', thickness=0.5, color=LIGHT_GREEN, spaceAfter=6))
    story.append(Paragraph('6.1 Mechanism', h2))
    story.append(Paragraph(
        'Reactive nitrogen applied to soils as synthetic fertilizers or manure undergoes '
        'microbial transformations that produce N<sub>2</sub>O as a by-product of both '
        'nitrification and denitrification. Globally, agricultural soils are the largest '
        'anthropogenic source of N<sub>2</sub>O, contributing ~3.8 Mt N<sub>2</sub>O-N '
        'annually (Tian et al., 2020). Two primary levers exist: (1) reducing excess nitrogen '
        'application through precision agriculture, and (2) slowing nitrogen transformation '
        'rates using nitrification inhibitors.', body))
    story.append(Paragraph('6.2 Precision Agriculture', h2))
    story.append(Paragraph(
        'Variable rate application (VRA) technologies, guided by remote sensing, soil sensor '
        'networks, and yield mapping, enable spatially optimised fertilizer placement that '
        'matches nitrogen supply to crop demand at sub-field resolution. Studies demonstrate '
        'that VRA can reduce nitrogen application rates by 10–25% without yield penalty, '
        'translating to proportional N<sub>2</sub>O reductions (Lark et al., 2021).', body))
    story.append(Paragraph('6.3 Nitrification Inhibitors', h2))
    story.append(Paragraph(
        'Compounds such as dicyandiamide (DCD) and 3,4-dimethylpyrazole phosphate (DMPP) '
        'inhibit the ammonia-oxidising bacteria responsible for the nitrification step '
        'of the nitrogen cycle. A global meta-analysis by Qiao et al. (2015) found that '
        'nitrification inhibitors reduce N<sub>2</sub>O emissions by 38% on average (range: '
        '20–55%) across soil types and climates. DMPP is stable at higher temperatures and '
        'demonstrates superior efficacy in tropical systems.', body))

    info_data2 = [[Paragraph(
        '<b>Combined approach:</b> Integrating precision nitrogen management with nitrification '
        'inhibitors can achieve N<sub>2</sub>O reductions of 20–50%, with additional co-benefits '
        'including improved nitrogen use efficiency, reduced nitrate leaching, and lower input costs.',
        body_small)]]
    info_table2 = Table(info_data2, colWidths=[W-4.4*cm])
    info_table2.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), colors.HexColor('#FFF8E7')),
        ('LEFTPADDING', (0,0), (-1,-1), 10),
        ('RIGHTPADDING', (0,0), (-1,-1), 10),
        ('TOPPADDING', (0,0), (-1,-1), 8),
        ('BOTTOMPADDING', (0,0), (-1,-1), 8),
        ('BOX', (0,0), (-1,-1), 1, colors.HexColor('#C47F17')),
        ('LINEAFTER', (0,0), (0,-1), 4, AMBER),
    ]))
    story.append(info_table2)
    story.append(PageBreak())

    # ── 7. AGROFORESTRY ───────────────────────────────────────────────────────
    story.append(Paragraph('7. Mitigation Strategy 5: Agroforestry and Carbon Sequestration', h1))
    story.append(HRFlowable(width='100%', thickness=0.5, color=LIGHT_GREEN, spaceAfter=6))
    story.append(Paragraph('7.1 Mechanism and Types', h2))
    story.append(Paragraph(
        'Agroforestry integrates trees and shrubs into crop and livestock systems, creating '
        'multi-functional landscapes that simultaneously produce food, fibre, and ecosystem '
        'services. Carbon sequestration occurs in above-ground biomass, root systems, and '
        'improved soil organic carbon (SOC). Major agroforestry systems include silvopasture '
        '(trees integrated with pasture and livestock), alley cropping (trees between crop '
        'rows), windbreaks/shelterbelts, and riparian buffers.', body))
    story.append(Paragraph('7.2 Carbon Sequestration Rates', h2))
    story.append(Paragraph(
        'Sequestration rates vary widely by system type, climate zone, tree species, and '
        'management intensity. Nair et al. (2009) report above-ground carbon stocks of '
        '2–90 Mg C ha<super>-1</super> for various tropical agroforestry systems. Temperate '
        'silvopasture systems typically sequester 0.5–3.0 Mg C ha<super>-1</super> yr<super>-1</super> '
        '(Kim et al., 2016). Across all system types, IPCC AR6 (2022) estimates a global '
        'technical mitigation potential of 2.1–2.6 Gt CO<sub>2</sub>e yr<super>-1</super> '
        'from agroforestry by 2050.', body))
    story.append(Paragraph('7.3 Co-benefits and Permanence', h2))
    story.append(Paragraph(
        'Agroforestry delivers exceptionally broad co-benefits: biodiversity enhancement, '
        'microclimate regulation, improved soil health, nitrogen cycling from leguminous '
        'trees, reduced erosion, and enhanced farm resilience to climate extremes. '
        'However, the permanence of sequestered carbon depends on long-term land management '
        'and is vulnerable to disturbances including fire, pests, and policy changes. '
        'Monitoring, Reporting, and Verification (MRV) frameworks are needed to ensure '
        'carbon credit integrity.', body))

    # ── 8. COMPARATIVE ANALYSIS ───────────────────────────────────────────────
    story.append(Paragraph('8. Comparative Analysis', h1))
    story.append(HRFlowable(width='100%', thickness=0.5, color=LIGHT_GREEN, spaceAfter=6))

    # insert chart
    chart_buf = make_chart()
    img = Image(chart_buf, width=W-4.4*cm, height=(W-4.4*cm)*0.56)
    story.append(img)
    story.append(Paragraph(
        'Figure 1: Comparative GHG emission reduction (%) across five major mitigation strategies. '
        'Error bars represent the published literature range. Bar color denotes primary gas targeted. '
        'Secondary co-benefit reductions shown where documented (sources: IPCC AR6, 2022; FAO, 2020; '
        'Beauchemin et al., 2020; Qiao et al., 2015; Nair et al., 2009).', caption))
    story.append(Spacer(1, 0.3*cm))

    # radar chart
    radar_buf = make_radar()
    radar_img = Image(radar_buf, width=10*cm, height=10*cm)
    radar_caption = Paragraph(
        'Figure 2: Multi-criteria performance radar comparing mitigation strategies across six dimensions '
        '(scale 0–10). Scores represent author synthesis of published literature.',
        caption)
    radar_wrap = Table([[radar_img, radar_caption]],
                        colWidths=[10.5*cm, W-4.4*cm-10.5*cm])
    radar_wrap.setStyle(TableStyle([
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('LEFTPADDING', (1,0), (1,0), 10),
    ]))
    story.append(radar_wrap)

    story.append(Paragraph(
        'Manure management systems (biogas) and AWD offer the highest absolute CH<sub>4</sub> '
        'reductions, while fertilizer optimization provides the most cost-effective N<sub>2</sub>O '
        'mitigation pathway. Agroforestry, despite moderate emission reduction percentages, uniquely '
        'delivers permanent atmospheric CO<sub>2</sub> removal and the broadest portfolio of '
        'co-benefits. Feed additives excel in technological maturity and farm-level ease of adoption.', body))

    # summary table
    story.append(Paragraph('Table 2: Summary Comparison of Mitigation Strategies', h3))
    t2_data = [
        ['Strategy', 'Primary Gas', 'Reduction Range', 'Cost (USD/t CO\u2082e)', 'Scalability'],
        ['Feed Additives', 'CH\u2084', '20–30%', '15–50', 'Medium-High'],
        ['Manure Biogas', 'CH\u2084', '50–80%', '20–60', 'Medium'],
        ['Alt. Wetting & Drying', 'CH\u2084', '30–70%', '1–15', 'High'],
        ['Fertilizer Optimization', 'N\u2082O', '20–50%', '5–30', 'High'],
        ['Agroforestry', 'CO\u2082', '15–50%', '5–50', 'Medium'],
    ]
    t2 = Table(t2_data, colWidths=[4.8*cm, 2.2*cm, 2.8*cm, 3.2*cm, 2.8*cm])
    t2.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), DARK_GREEN),
        ('TEXTCOLOR',  (0,0), (-1,0), WHITE),
        ('FONTNAME',   (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTSIZE',   (0,0), (-1,-1), 8.5),
        ('ROWBACKGROUNDS', (0,1), (-1,-1), [WHITE, PALE_GREEN]),
        ('GRID', (0,0), (-1,-1), 0.4, MID_GRAY),
        ('ALIGN', (1,0), (-1,-1), 'CENTER'),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('TOPPADDING', (0,0), (-1,-1), 5),
        ('BOTTOMPADDING', (0,0), (-1,-1), 5),
    ]))
    story.append(t2)
    story.append(Paragraph('Sources: IPCC AR6 (2022); Smith et al. (2014); FAO (2020)', caption))
    story.append(PageBreak())

    # ── 9. CHALLENGES ─────────────────────────────────────────────────────────
    story.append(Paragraph('9. Challenges and Barriers to Implementation', h1))
    story.append(HRFlowable(width='100%', thickness=0.5, color=LIGHT_GREEN, spaceAfter=6))

    challenges = [
        ('Economic barriers', 'Most mitigation strategies require upfront capital investment with '
         'payback periods that extend beyond typical farmer planning horizons. Smallholder farmers '
         'in developing countries — who manage a large share of global agricultural land — often '
         'lack access to credit or risk-sharing mechanisms.'),
        ('Knowledge and extension gaps', 'Adoption of precision agriculture, feed additives, and '
         'AWD requires technical knowledge, monitoring capacity, and behaviour change. Extension '
         'service networks are underfunded in many regions, creating implementation gaps.'),
        ('Measurement and verification', 'Reliable quantification of on-farm GHG reductions is '
         'technically complex and expensive. Standardised MRV protocols are required to support '
         'carbon markets and compliance mechanisms.'),
        ('Policy and regulatory environment', 'Regulatory approval for novel feed additives '
         '(e.g., 3-NOP, Asparagopsis) varies across jurisdictions. Carbon credit frameworks for '
         'agroforestry and soils lack harmonisation, limiting market access.'),
        ('Trade-offs and rebound effects', 'Some strategies involve trade-offs — AWD may '
         'increase N\u2082O slightly; fertilizer reduction may reduce yields if poorly managed. '
         'Careful agronomic management is essential to capture net benefits.'),
    ]
    for title_c, text_c in challenges:
        story.append(Paragraph(f'<b>{title_c}:</b> {text_c}', body))

    # ── 10. POLICY RECOMMENDATIONS ────────────────────────────────────────────
    story.append(Paragraph('10. Policy Recommendations', h1))
    story.append(HRFlowable(width='100%', thickness=0.5, color=LIGHT_GREEN, spaceAfter=6))
    story.append(Paragraph(
        'Evidence from this analysis supports the following policy priorities:', body))

    recs = [
        'Integrate agricultural GHG mitigation into Nationally Determined Contributions (NDCs) '
        'with measurable, time-bound targets for each strategy.',
        'Establish results-based payment schemes and green credit lines to reduce financial '
        'barriers for smallholder adoption of AWD, precision fertilization, and agroforestry.',
        'Accelerate regulatory harmonisation for feed additives, enabling scale-up of '
        '3-NOP and seaweed-based supplements across major livestock-producing nations.',
        'Fund national and regional MRV infrastructure for agricultural carbon to underpin '
        'carbon market integrity and facilitate payments for ecosystem services.',
        'Support knowledge transfer and extension services, particularly in South and Southeast '
        'Asia and Sub-Saharan Africa, to operationalise AWD and fertilizer best-practice at scale.',
        'Invest in cross-strategy research to optimise co-deployment of complementary approaches, '
        'recognising that no single strategy is sufficient to meet 1.5°C-aligned targets.',
    ]
    for i, r in enumerate(recs, 1):
        row_data = [[Paragraph(str(i), body_small), Paragraph(r, body_small)]]
        rec_table = Table(row_data, colWidths=[0.6*cm, W-5.0*cm])
        rec_table.setStyle(TableStyle([
            ('VALIGN', (0,0), (-1,-1), 'TOP'),
            ('LEFTPADDING', (0,0), (-1,-1), 4),
            ('TOPPADDING', (0,0), (-1,-1), 3),
            ('BOTTOMPADDING', (0,0), (-1,-1), 3),
            ('BACKGROUND', (0,0), (0,0), PALE_GREEN),
            ('FONTNAME', (0,0), (0,0), 'Helvetica-Bold'),
        ]))
        story.append(rec_table)
    story.append(Spacer(1, 0.4*cm))

    # ── 11. CONCLUSION ────────────────────────────────────────────────────────
    story.append(Paragraph('11. Conclusion', h1))
    story.append(HRFlowable(width='100%', thickness=0.5, color=LIGHT_GREEN, spaceAfter=6))
    story.append(Paragraph(
        'Agriculture stands at a critical juncture: it must simultaneously reduce its GHG footprint '
        'and increase food production under a changing climate. This assignment has examined five '
        'major mitigation strategies, each targeting distinct emission sources and gases. Feed '
        'additives and AWD offer rapid, scalable CH<sub>4</sub> reductions with relatively low '
        'costs. Manure biogas systems provide high-impact CH<sub>4</sub> capture alongside '
        'renewable energy co-benefits. Fertilizer optimization, particularly the integration of '
        'precision management with nitrification inhibitors, offers the primary route to meaningful '
        'N<sub>2</sub>O reduction. Agroforestry provides durable CO<sub>2</sub> sequestration '
        'alongside the broadest suite of ecological co-benefits.', body))
    story.append(Paragraph(
        'No single strategy is sufficient. Achieving Paris Agreement-aligned emissions trajectories '
        'from the agricultural sector will require the co-deployment of multiple interventions, '
        'supported by enabling policy frameworks, financial mechanisms, and robust monitoring '
        'infrastructure. The technical potential is clearly established in the literature; '
        'translating this potential into real-world impact remains the defining challenge '
        'for agricultural scientists, policymakers, and farming communities in the coming decade.', body))

    # ── 12. REFERENCES ────────────────────────────────────────────────────────
    story.append(Paragraph('12. References', h1))
    story.append(HRFlowable(width='100%', thickness=0.5, color=LIGHT_GREEN, spaceAfter=6))

    refs = [
        'Amon, B., Kryvoruchko, V., Amon, T., & Zechmeister-Boltenstern, S. (2006). Methane, nitrous oxide and ammonia emissions during storage and after application of dairy cattle slurry and influence of slurry treatment. <i>Agriculture, Ecosystems & Environment</i>, 112(2–3), 153–162.',
        'Beauchemin, K. A., Ungerfeld, E. M., Eckard, R. J., & Wang, M. (2020). Review: Fifty years of research on rumen methanogenesis: lessons learned and future challenges for mitigation. <i>Animal</i>, 14(S1), s2–s16.',
        'FAO. (2020). <i>Methane emissions in livestock and rice systems: sources, quantification, mitigation and metrics</i>. Food and Agriculture Organization of the United Nations, Rome.',
        'Gerber, P. J., Steinfeld, H., Henderson, B., Mottet, A., Opio, C., Dijkman, J., … & Tempio, G. (2013). <i>Tackling climate change through livestock: a global assessment of emissions and mitigation opportunities</i>. FAO, Rome.',
        'Hristov, A. N., Oh, J., Giallongo, F., Frederick, T. W., Harper, M. T., Weeks, H. L., … & Ghosh, S. (2015). An inhibitor persistently decreased enteric methane emission from dairy cows with no negative effect on milk production. <i>Proceedings of the National Academy of Sciences</i>, 112(34), 10663–10668.',
        'IPCC. (2022). <i>Climate Change 2022: Mitigation of Climate Change. Contribution of Working Group III to the Sixth Assessment Report</i>. Cambridge University Press. doi:10.1017/9781009157926.',
        'Kim, D. G., Kirschbaum, M. U. F., & Beedy, T. L. (2016). Carbon sequestration and net emissions of CH<sub>4</sub> and N<sub>2</sub>O under agroforestry. <i>Agriculture, Ecosystems & Environment</i>, 226, 107–118.',
        'Kritee, K., Nair, D., Zavala-Araiza, D., Proville, J., Rudek, J., Adhya, T. K., … & Hamburg, S. P. (2018). High nitrous oxide fluxes from rice indicate the need to manage water for climate mitigation and adaptation. <i>Proceedings of the National Academy of Sciences</i>, 115(39), 9720–9725.',
        'Lagomarsino, A., Agnelli, A. E., Pastorelli, R., Pallara, G., Rasse, D. P., & Silvennoinen, H. (2016). Past water management affected greenhouse gas emissions and microbial community pattern in Italian rice paddy soils. <i>Soil Biology and Biochemistry</i>, 93, 17–27.',
        'Lark, T. J., Mueller, R. M., Johnson, D. M., & Gibbs, H. K. (2021). Measuring the cost of US cropland expansion for N<sub>2</sub>O emissions. <i>Environmental Research Letters</i>, 16(8), 085001.',
        'Linquist, B. A., Adviento-Borbe, M. A., Pittelkow, C. M., van Kessel, C., & van Groenigen, K. J. (2015). Fertilizer management practices and greenhouse gas emissions from rice systems. <i>Global Change Biology</i>, 18(1), 394–406.',
        'Nair, P. K. R., Kumar, B. M., & Nair, V. D. (2009). Agroforestry as a strategy for carbon sequestration. <i>Journal of Plant Nutrition and Soil Science</i>, 172(1), 10–23.',
        'Qiao, C., Liu, L., Hu, S., Compton, J. E., Greaver, T. L., & Li, Q. (2015). How inhibiting nitrification affects nitrogen cycle and reduces environmental impacts of anthropogenic nitrogen input. <i>Global Change Biology</i>, 21(3), 1249–1257.',
        'Saunois, M., Stavert, A. R., Poulter, B., Bousquet, P., Canadell, J. G., Jackson, R. B., … & Zhuang, Q. (2020). The global methane budget 2000–2017. <i>Earth System Science Data</i>, 12(3), 1561–1623.',
        'Smith, P., Bustamante, M., Ahammad, H., Clark, H., Dong, H., Elsiddig, E. A., … & Tubiello, F. N. (2014). Agriculture, Forestry and Other Land Use (AFOLU). In: <i>Climate Change 2014: Mitigation of Climate Change. IPCC Fifth Assessment Report</i>. Cambridge University Press.',
        'Tian, H., Xu, R., Canadell, J. G., Thompson, R. L., Winiwarter, W., Suntharalingam, P., … & Yao, Y. (2020). A comprehensive quantification of global nitrous oxide sources and sinks. <i>Nature</i>, 586(7828), 248–256.',
        'Weiland, P. (2010). Biogas production: current state and perspectives. <i>Applied Microbiology and Biotechnology</i>, 85(4), 849–860.',
    ]
    for r in refs:
        story.append(Paragraph(r, ref_style))

    doc.build(story, onFirstPage=on_page, onLaterPages=on_page)
    print('Done:', out)

build()


LayoutError: Flowable <CoverFlowable at 0x7dd40d8105f0 frame=normal>...(595.2755905511812 x 841.8897637795277) too large on page 2 in frame 'normal'(458.5511811023622 x 705.1653543307087*) of template 'Later'